In [ ]:
from IPython.display import display, Markdown

def solve(z, y, mu_z, Sigma_z, W, b, Sigma_y):
    """
    Display symbolic results for a linear Gaussian system.

    Parameters — all strings, use whatever LaTeX notation you like:
        z        : latent variable name,       e.g. r'\mathbf{x}_1'
        y        : observed variable name,     e.g. r'\mathbf{x}_2'
        mu_z     : prior mean,                 e.g. r'\mathbf{0}'
        Sigma_z  : prior covariance,           e.g. r'\boldsymbol{\Sigma}'
        W        : weight matrix,              e.g. r'\mathbf{A}'
        b        : bias vector,                e.g. r'\mathbf{0}'
        Sigma_y  : likelihood covariance,      e.g. r'\boldsymbol{\Sigma}'
    """
    zero_vals = (r'0', r'\mathbf{0}', r'\boldsymbol{0}')

    def mul(A, x):
        """'A x', but simplify to 0 if either factor is zero."""
        if A in zero_vals or x in zero_vals:
            return r'\mathbf{0}'
        return f'{A} {x}'

    def add(a, b_):
        """'a + b_', but drop the '+ b_' term if b_ is zero."""
        if b_ in zero_vals:
            return a
        return f'{a} + {b_}'

    def paren(expr):
        """Wrap expr in \\left(...\\right) if it is a compound expression (contains + or -)."""
        if any(op in expr for op in ('+', '-')):
            return rf'\left( {expr} \right)'
        return expr

    # Derived expressions
    Sigma_z_p = paren(Sigma_z)  # parenthesised if compound
    Sigma_y_p = paren(Sigma_y)
    likelihood_mean = add(f'{W}{z}', b)
    W_muz     = mul(W, mu_z)
    mu_y      = W_muz if b in zero_vals else f'{W_muz} + {b}'
    cov_yy    = f'{Sigma_y_p} + {W} {Sigma_z_p} {W}^\\top'
    cov_zy    = f'{Sigma_z_p} {W}^\\top'
    cov_yz    = f'{W} {Sigma_z_p}'
    prec_post = f'{Sigma_z_p}^{{-1}} + {W}^\\top {Sigma_y_p}^{{-1}} {W}'
    cov_post  = rf'\left( {prec_post} \right)^{{-1}}'

    Sz_inv_muz = mul(f'{Sigma_z_p}^{{-1}}', mu_z)
    if b in zero_vals:
        inner = (f'{W}^\\top {Sigma_y}^{{-1}} {y}'
                 if Sz_inv_muz in zero_vals else
                 f'{W}^\\top {Sigma_y}^{{-1}} {y} + {Sz_inv_muz}')
    else:
        inner = (f'{W}^\\top {Sigma_y}^{{-1}} ({y} - {b})'
                 if Sz_inv_muz in zero_vals else
                 f'{W}^\\top {Sigma_y}^{{-1}} ({y} - {b}) + {Sz_inv_muz}')
    mu_post = rf'\Sigma_{{{z}|{y}}} \left[ {inner} \right]'

    display(Markdown('### Given'))
    display(Markdown(rf"""
$$\begin{{align}}
p({z}) &= \mathcal{{N}}\left({z} \mid {mu_z}, {Sigma_z}\right) \\\\
p({y} \mid {z}) &= \mathcal{{N}}\left({y} \mid {W}{z} + {b}, {Sigma_y}\right)
\end{{align}}$$
"""))

    display(Markdown(f'### Joint $p({z}, {y})$'))
    display(Markdown(rf"""
$$\begin{{pmatrix}} {z} \\\\ {y} \end{{pmatrix}}
\sim \mathcal{{N}}\left(
  \begin{{pmatrix}} {mu_z} \\\\ {mu_y} \end{{pmatrix}},
  \begin{{pmatrix}}
    {Sigma_z} & {cov_zy} \\\\
    {cov_yz}  & {cov_yy}
  \end{{pmatrix}}
\right)$$
"""))

    display(Markdown(f'### Marginal $p({y})$'))
    display(Markdown(rf"""
$$p({y}) = \mathcal{{N}}\left({y} \mid {mu_y}, {cov_yy}\right)$$
"""))

    display(Markdown(f'### Posterior $p({z} \\mid {y})$ — Bayes rule for Gaussians'))
    display(Markdown(rf"""
$$\begin{{align}}
p({z} \mid {y}) &= \mathcal{{N}}\left({z} \mid \mu_{{{z}|{y}}}, \Sigma_{{{z}|{y}}}\right) \\\\
\Sigma_{{{z}|{y}}} &= {cov_post} \\\\
\mu_{{{z}|{y}}} &= {mu_post}
\end{{align}}$$
"""))


<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
C:\Users\NTres\AppData\Local\Temp\ipykernel_33280\3848766484.py:8: SyntaxWarning: invalid escape sequence '\m'
  z        : latent variable name,       e.g. r'\mathbf{x}_1'


In [3]:
# ── Example: Question 1.1 ────────────────────────────────────────────────
# Markov chain:  x1 ~ N(0, Σ),  x2|x1 ~ N(A x1, Σ)
# Find p(x1 | x2)

solve(
    z       = r'\mathbf{\theta}',
    y       = r'\mathbf{y}',
    mu_z    = r'\mathbf{-m}',
    Sigma_z = r'\tau^2 \boldsymbol{I}',
    W       = r'\mathbf{X}',
    b       = r'\mathbf{0}',
    Sigma_y = r'\sigma^2 \boldsymbol{I}',
)

### Given


$$\begin{align}
p(\mathbf{\theta}) &= \mathcal{N}\left(\mathbf{\theta} \mid \mathbf{-m}, \tau^2 \boldsymbol{I}\right) \\\\
p(\mathbf{y} \mid \mathbf{\theta}) &= \mathcal{N}\left(\mathbf{y} \mid \mathbf{X}\mathbf{\theta} + \mathbf{0}, \sigma^2 \boldsymbol{I}\right)
\end{align}$$


### Joint $p(\mathbf{\theta}, \mathbf{y})$


$$\begin{pmatrix} \mathbf{\theta} \\\\ \mathbf{y} \end{pmatrix}
\sim \mathcal{N}\left(
  \begin{pmatrix} \mathbf{-m} \\\\ \mathbf{X} \mathbf{-m} \end{pmatrix},
  \begin{pmatrix}
    \tau^2 \boldsymbol{I} & \tau^2 \boldsymbol{I} \mathbf{X}^\top \\\\
    \mathbf{X} \tau^2 \boldsymbol{I}  & \sigma^2 \boldsymbol{I} + \mathbf{X} \tau^2 \boldsymbol{I} \mathbf{X}^\top
  \end{pmatrix}
\right)$$


### Marginal $p(\mathbf{y})$


$$p(\mathbf{y}) = \mathcal{N}\left(\mathbf{y} \mid \mathbf{X} \mathbf{-m}, \sigma^2 \boldsymbol{I} + \mathbf{X} \tau^2 \boldsymbol{I} \mathbf{X}^\top\right)$$


### Posterior $p(\mathbf{\theta} \mid \mathbf{y})$ — Bayes rule for Gaussians


$$\begin{align}
p(\mathbf{\theta} \mid \mathbf{y}) &= \mathcal{N}\left(\mathbf{\theta} \mid \mu_{\mathbf{\theta}|\mathbf{y}}, \Sigma_{\mathbf{\theta}|\mathbf{y}}\right) \\\\
\Sigma_{\mathbf{\theta}|\mathbf{y}} &= \left( \tau^2 \boldsymbol{I}^{-1} + \mathbf{X}^\top \sigma^2 \boldsymbol{I}^{-1} \mathbf{X} \right)^{-1} \\\\
\mu_{\mathbf{\theta}|\mathbf{y}} &= \Sigma_{\mathbf{\theta}|\mathbf{y}} \left[ \mathbf{X}^\top \sigma^2 \boldsymbol{I}^{-1} \mathbf{y} + \tau^2 \boldsymbol{I}^{-1} \mathbf{-m} \right]
\end{align}$$


In [ ]:
from IPython.display import display, Markdown


def solve_mixture(z, y, components, W, b, Sigma_y):
    """
    Display symbolic results for a linear Gaussian system with a Gaussian
    mixture prior on the latent variable z.

    Parameters — all strings, use whatever LaTeX notation you like:
        z          : latent variable name,  e.g. r'\\theta'
        y          : observed variable name, e.g. r'y'
        components : list of (weight, mu_z, Sigma_z) tuples, e.g.
                         [
                           (r'\\frac{1}{2}', r'm_1', r'v_1'),
                           (r'\\frac{1}{2}', r'm_2', r'v_2'),
                         ]
                     Weights are treated as symbolic strings; they do not
                     need to sum to 1 numerically.
        W          : weight matrix,          e.g. r'X'
        b          : bias vector,            e.g. r'0'
        Sigma_y    : likelihood covariance,  e.g. r'v_y'

    Also works as a drop-in for a single-component mixture (just pass one
    tuple), which reproduces the same output as the original solve().
    """

    zero_vals = (r'0', r'\mathbf{0}', r'\boldsymbol{0}')

    def extract_sign(expr):
        """Return (sign, expr_without_sign) where sign is '-' or ''.
        Handles bare '-x' and also '\\cmd{-x}' style LaTeX wrappers."""
        import re
        # bare minus:  -x
        if expr.startswith('-'):
            return '-', expr[1:].lstrip()
        # minus inside a LaTeX command: \cmd{-...}  e.g. \boldsymbol{-m}
        m = re.match(r'^(\\[a-zA-Z]+\{)-(.+)(\})$', expr)
        if m:
            return '-', m.group(1) + m.group(2) + m.group(3)
        return '', expr

    def mul(A, x):
        if A in zero_vals or x in zero_vals:
            return r'\mathbf{0}'
        # Pull a leading minus out of either factor so the sign sits in front:
        # e.g.  X \boldsymbol{-m}  ->  -X \boldsymbol{m}
        s1, A_ = extract_sign(A)
        s2, x_ = extract_sign(x)
        sign = '-' if (s1 == '-') != (s2 == '-') else ''
        return f'{sign}{A_} {x_}'

    def add(a, b_):
        if b_ in zero_vals:
            return a
        return f'{a} + {b_}'

    def paren(expr):
        if any(op in expr for op in ('+', '-')):
            return rf'\left( {expr} \right)'
        return expr

    K = len(components)
    single = (K == 1)

    def inv(expr):
        """Return {expr}^{-1}, safely braced to avoid double-superscript."""
        return rf'{{{expr}}}^{{-1}}'

    # ------------------------------------------------------------------ #
    # Pre-compute per-component quantities                                 #
    # ------------------------------------------------------------------ #
    Sigma_y_p = paren(Sigma_y)

    comp_data = []
    for k, (pi_k, mu_k, Sigma_k) in enumerate(components, start=1):
        idx = '' if single else f'_{k}'           # subscript suffix
        Sigma_k_p = paren(Sigma_k)

        # --- marginal p(y) for component k ---
        W_mu_k   = mul(W, mu_k)
        mu_y_k   = W_mu_k if b in zero_vals else f'{W_mu_k} + {b}'
        cov_yy_k = f'{Sigma_y_p} + {W} {Sigma_k_p} {W}^\\top'

        # --- posterior covariance ---
        # {…}^{-1} bracing avoids double-superscript for tokens like \sigma_0^2

        prec_post_k = (f'{inv(Sigma_k_p)} + '
                       f'{W}^\\top {inv(Sigma_y_p)} {W}')
        cov_post_k  = rf'\left( {prec_post_k} \right)^{{-1}}'

        # --- posterior mean inner bracket ---
        Sz_inv_muz_k = mul(inv(Sigma_k_p), mu_k)
        Sy_inv = inv(Sigma_y_p)
        if b in zero_vals:
            inner_k = (
                f'{W}^\\top {Sy_inv} {y}'
                if Sz_inv_muz_k in zero_vals else
                f'{W}^\\top {Sy_inv} {y} + {Sz_inv_muz_k}'
            )
        else:
            inner_k = (
                f'{W}^\\top {Sy_inv} ({y} - {b})'
                if Sz_inv_muz_k in zero_vals else
                f'{W}^\\top {Sy_inv} ({y} - {b}) + {Sz_inv_muz_k}'
            )

        cov_post_label  = rf'\Sigma_{{{z}|{y}{idx}}}'
        mean_post_label = rf'\mu_{{{z}|{y}{idx}}}'
        mu_post_k = rf'{cov_post_label} \left[ {inner_k} \right]'

        comp_data.append(dict(
            k=k, pi=pi_k, mu_z=mu_k, Sigma_z=Sigma_k,
            Sigma_z_p=Sigma_k_p,
            mu_y=mu_y_k, cov_yy=cov_yy_k,
            cov_zy=f'{Sigma_k_p} {W}^\\top',
            cov_yz=f'{W} {Sigma_k_p}',
            cov_post=cov_post_k,
            cov_post_label=cov_post_label,
            mean_post_label=mean_post_label,
            mu_post=mu_post_k,
        ))

    # ------------------------------------------------------------------ #
    # 1.  Given                                                            #
    # ------------------------------------------------------------------ #
    display(Markdown('### Given'))

    if single:
        pi, mu_k, Sigma_k = components[0]
        display(Markdown(rf"""
$$\begin{{align}}
p({z}) &= \mathcal{{N}}\left({z} \mid {mu_k},\, {Sigma_k}\right) \\\\
p({y} \mid {z}) &= \mathcal{{N}}\left({y} \mid {W}{z} + {b},\, {Sigma_y}\right)
\end{{align}}$$
"""))
    else:
        # Prior: mixture
        terms = r' + '.join(
            rf'{c["pi"]}\,\mathcal{{N}}\!\left({z} \mid {c["mu_z"]},\, {c["Sigma_z"]}\right)'
            for c in comp_data
        )
        display(Markdown(rf"""
$$\begin{{align}}
p({z}) &= {terms} \\\\
p({y} \mid {z}) &= \mathcal{{N}}\left({y} \mid {W}{z} + {b},\, {Sigma_y}\right)
\end{{align}}$$
"""))

    # ------------------------------------------------------------------ #
    # 2.  Per-component joint & marginal                                   #
    # ------------------------------------------------------------------ #
    section = '### Joint and marginal per component' if not single else f'### Joint $p({z}, {y})$'
    display(Markdown(section))

    for c in comp_data:
        idx = '' if single else f'_{c["k"]}'
        header = '' if single else rf'**Component {c["k"]}** $\;(\pi_{c["k"]} = {c["pi"]})$'
        if header:
            display(Markdown(header))

        # Joint
        display(Markdown(rf"""
$$\begin{{pmatrix}} {z} \\\\ {y} \end{{pmatrix}}
\sim \mathcal{{N}}\!\left(
  \begin{{pmatrix}} {c["mu_z"]} \\\\ {c["mu_y"]} \end{{pmatrix}},\;
  \begin{{pmatrix}}
    {c["Sigma_z"]} & {c["cov_zy"]} \\\\
    {c["cov_yz"]}  & {c["cov_yy"]}
  \end{{pmatrix}}
\right)$$
"""))

    # ------------------------------------------------------------------ #
    # 3.  Marginal p(y)                                                    #
    # ------------------------------------------------------------------ #
    display(Markdown(f'### Marginal $p({y})$'))

    if single:
        c = comp_data[0]
        display(Markdown(rf"""
$$p({y}) = \mathcal{{N}}\!\left({y} \mid {c["mu_y"]},\; {c["cov_yy"]}\right)$$
"""))
    else:
        # Build an align block:  p(y) = term_1  (first line)
        #                              + term_2  (subsequent lines, indented)
        first = comp_data[0]
        rest  = comp_data[1:]
        first_line = (rf'p({y}) &= {first["pi"]}\,\mathcal{{N}}\!\left('
                      rf'{y} \mid {first["mu_y"]},\; {first["cov_yy"]}\right)')
        extra_lines = '\n'.join(
            rf'       &\quad +\; {c["pi"]}\,\mathcal{{N}}\!\left('
            rf'{y} \mid {c["mu_y"]},\; {c["cov_yy"]}\right) \\\\'
            for c in rest
        )
        align_body = first_line + r' \\' + ('\n' + extra_lines if extra_lines else '')
        display(Markdown(rf"""
$$\begin{{align}}
{align_body}
\end{{align}}$$
"""))
        # Note on updated weights
        r_terms = r',\quad '.join(
            rf'r_{c["k"]} \;\propto\; {c["pi"]}\,\mathcal{{N}}\!\left({y} \mid {c["mu_y"]},\; {c["cov_yy"]}\right)'
            for c in comp_data
        )
        display(Markdown(
            r'> **Posterior mixture weights** (responsibilities after observing $' + y + r'$):'
            + '\n>\n> $$' + r_terms + r'$$'
        ))

    # ------------------------------------------------------------------ #
    # 4.  Posterior p(z | y)                                               #
    # ------------------------------------------------------------------ #
    display(Markdown(
        f'### Posterior $p({z} \\mid {y})$'
        + ('' if single else ' — weighted mixture of Gaussians')
    ))

    if single:
        c = comp_data[0]
        display(Markdown(rf"""
$$\begin{{align}}
p({z} \mid {y}) &= \mathcal{{N}}\!\left({z} \mid \mu_{{{z}|{y}}},\; \Sigma_{{{z}|{y}}}\right) \\\\
\Sigma_{{{z}|{y}}} &= {c["cov_post"]} \\\\
\mu_{{{z}|{y}}} &= {c["mu_post"]}
\end{{align}}$$
"""))
    else:
        # Display each component's posterior parameters
        post_terms = r' + '.join(
            rf'r_{c["k"]}\,\mathcal{{N}}\!\left({z} \mid {c["mean_post_label"]},\; {c["cov_post_label"]}\right)'
            for c in comp_data
        )
        display(Markdown(rf'$$p({z} \mid {y}) = {post_terms}$$'))
        display(Markdown('where for each component $k$:'))

        align_rows = []
        for c in comp_data:
            align_rows.append(
                rf'{c["cov_post_label"]} &= {c["cov_post"]} \\\\'
            )
            align_rows.append(
                rf'{c["mean_post_label"]} &= {c["mu_post"]} \\\\'
            )
        align_body = '\n'.join(align_rows).rstrip(r'\\')
        display(Markdown(rf"""
$$\begin{{align}}
{align_body}
\end{{align}}$$
"""))


# # ------------------------------------------------------------------ #
# # Quick smoke-test (run this cell to verify both call styles work)   #
# # ------------------------------------------------------------------ #
# if __name__ == '__main__':

#     print("=== Single-component (reproduces original solve behaviour) ===")
#     solve_mixture(
#         z=r'\theta', y=r'y',
#         components=[(r'1', r'\mu_0', r'\sigma_0^2')],
#         W=r'X', b=r'0', Sigma_y=r'\sigma_y^2',
#     )

#     print("\n=== Two-component mixture (the new case) ===")
#     solve_mixture(
#         z=r'\theta', y=r'y',
#         components=[
#             (r'\tfrac{1}{2}', r'm_1', r'v_1'),
#             (r'\tfrac{1}{2}', r'm_2', r'v_2'),
#         ],
#         W=r'X', b=r'0', Sigma_y=r'v_y',
#     )

print("\n=== Two-component mixture (the new case) ===")
solve_mixture(
    z=r'\theta', y=r'y',
    components=[
        (r'\tfrac{1}{2}', r'\boldsymbol{-m}', r'\tau^2 \boldsymbol{I}'),
        (r'\tfrac{1}{2}', r'\boldsymbol{m}', r'\tau^2 \boldsymbol{I}'),
    ],
    W=r'X', b=r'0', Sigma_y=r'\sigma^2 \boldsymbol{I}',
)

In [13]:
print("\n=== Two-component mixture (the new case) ===")
solve_mixture(
    z=r'\theta', y=r'y',
    components=[
        (r'\tfrac{1}{2}', r'\boldsymbol{-m}', r'\tau^2 \boldsymbol{I}'),
        (r'\tfrac{1}{2}', r'\boldsymbol{m}', r'\tau^2 \boldsymbol{I}'),
    ],
    W=r'X', b=r'0', Sigma_y=r'\sigma^2 \boldsymbol{I}',
)


=== Two-component mixture (the new case) ===


### Given


$$\begin{align}
p(\theta) &= \tfrac{1}{2}\,\mathcal{N}\!\left(\theta \mid \boldsymbol{-m},\, \tau^2 \boldsymbol{I}\right) + \tfrac{1}{2}\,\mathcal{N}\!\left(\theta \mid \boldsymbol{m},\, \tau^2 \boldsymbol{I}\right) \\\\
p(y \mid \theta) &= \mathcal{N}\left(y \mid X\theta + 0,\, \sigma^2 \boldsymbol{I}\right)
\end{align}$$


### Joint and marginal per component

**Component 1** $\;(\pi_1 = \tfrac{1}{2})$

...

> **Posterior mixture weights** (responsibilities after observing $y$):
>
> $$r_1 \;\propto\; \tfrac{1}{2}\,\mathcal{N}\!\left(y \mid -X \boldsymbol{m},\; \sigma^2 \boldsymbol{I} + X \tau^2 \boldsymbol{I} X^\top\right),\quad r_2 \;\propto\; \tfrac{1}{2}\,\mathcal{N}\!\left(y \mid X \boldsymbol{m},\; \sigma^2 \boldsymbol{I} + X \tau^2 \boldsymbol{I} X^\top\right)$$

### Posterior $p(\theta \mid y)$ — weighted mixture of Gaussians

$$p(\theta \mid y) = r_1\,\mathcal{N}\!\left(\theta \mid \mu_{\theta|y_1},\; \Sigma_{\theta|y_1}\right) + r_2\,\mathcal{N}\!\left(\theta \mid \mu_{\theta|y_2},\; \Sigma_{\theta|y_2}\right)$$

where for each component $k$:


$$\begin{align}
\Sigma_{\theta|y_1} &= \left( {\tau^2 \boldsymbol{I}}^{-1} + X^\top {\sigma^2 \boldsymbol{I}}^{-1} X \right)^{-1} \\\\
\mu_{\theta|y_1} &= \Sigma_{\theta|y_1} \left[ X^\top {\sigma^2 \boldsymbol{I}}^{-1} y + -{\tau^2 \boldsymbol{I}}^{-1} \boldsymbol{m} \right] \\\\
\Sigma_{\theta|y_2} &= \left( {\tau^2 \boldsymbol{I}}^{-1} + X^\top {\sigma^2 \boldsymbol{I}}^{-1} X \right)^{-1} \\\\
\mu_{\theta|y_2} &= \Sigma_{\theta|y_2} \left[ X^\top {\sigma^2 \boldsymbol{I}}^{-1} y + {\tau^2 \boldsymbol{I}}^{-1} \boldsymbol{m} \right] 
\end{align}$$
